# 00. Metadata, FASTQ validation and MiXCR preprocessing

This notebook does only three things:

1. validates sample metadata and FASTQ pairs;
2. runs MiXCR for each library in bounded parallel mode;
3. creates a compact library QC table.

Biological hypothesis testing is performed in the next notebook.

In [ ]:

from pathlib import Path
import os, re, json, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path(os.environ.get("MEASLES_REPO", Path.cwd())).resolve()
MANIFEST = ROOT / "config" / "sample_manifest.csv"
KNOWN_ISSUES = ROOT / "config" / "metadata_known_issues.csv"
NEW_DIR = Path(os.environ.get("MEASLES_NEW_FASTQ_DIR", "/projects/mice_transplant_2025/raw/20260817_161454/Fastq")).expanduser()
legacy_dir_value = os.environ.get("MEASLES_LEGACY_FASTQ_DIR", "").strip()
LEGACY_DIR = Path(legacy_dir_value).expanduser() if legacy_dir_value else None
PRESET = os.environ.get("MEASLES_MIXCR_PRESET", "milab-human-dna-tcr-multiplex")
CPUS = int(os.environ.get("MEASLES_CPUS", os.environ.get("SLURM_CPUS_PER_TASK", "1")))
THREADS_PER_SAMPLE = int(os.environ.get("MEASLES_THREADS_PER_SAMPLE", "4"))
PILOT = os.environ.get("MEASLES_PILOT", "0") == "1"

for p in [ROOT/"results", ROOT/"figures", ROOT/"work"/"inputs", ROOT/"results"/"mixcr", ROOT/"results"/"clonotypes"]:
    p.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST)
for col in ["include", "is_control"]:
    if col in manifest.columns:
        manifest[col] = manifest[col].astype(str).str.lower().isin(["true", "1", "yes"])
manifest.head()


## Metadata checks

The source document states that the numeric prefix of a new FASTQ filename identifies the biological record. The current child/adult lists are incomplete, therefore unresolved age is kept as `unknown` instead of being guessed.

In [ ]:

issues = pd.read_csv(KNOWN_ISSUES)

# Automatic metadata checks
auto = []

dup_mask = manifest.duplicated(["patient_id", "analysis_timepoint", "batch"], keep=False)
for _, r in manifest[dup_mask].iterrows():
    auto.append({
        "library_id": r.library_id,
        "raw_sample_id": r.raw_sample_id,
        "issue": "duplicate_patient_timepoint",
        "note": "Multiple libraries map to the same patient/timepoint within a batch; they will be aggregated for biological analysis."
    })

for _, r in manifest[manifest["age_group"].eq("unknown")].iterrows():
    auto.append({
        "library_id": r.library_id,
        "raw_sample_id": r.raw_sample_id,
        "issue": "unknown_age",
        "note": "Patient is absent from the current child/adult lists; retained for QC and excluded from age-stratified tests."
    })

bad_tp = manifest[
    (~manifest["is_control"].astype(bool)) &
    (~manifest["analysis_timepoint"].fillna(-1).isin([1,2,3,4]))
]
for _, r in bad_tp.iterrows():
    auto.append({
        "library_id": r.library_id,
        "raw_sample_id": r.raw_sample_id,
        "issue": "invalid_timepoint",
        "note": "Measles sample has an invalid or missing analysis timepoint."
    })

issues = pd.concat([issues, pd.DataFrame(auto)], ignore_index=True).drop_duplicates()
issues.to_csv(ROOT/"results"/"metadata_issues.csv", index=False)

print("Libraries:", len(manifest))
print(manifest.groupby(["batch","age_group","disease_status"]).size())
print("\nMetadata issues:", len(issues))
issues.head(20)


## FASTQ discovery

New libraries are discovered by the integer prefix (`10`, `11`, ...). Legacy libraries are discovered by `UDP0097` to `UDP0112`.

All discovered FASTQ pairs are linked into one standardized working directory. This keeps the actual raw files untouched and lets MiXCR process multiple lanes directly.

In [ ]:

def discover_new(index):
    if not NEW_DIR.exists():
        return []
    r1 = sorted(NEW_DIR.rglob(f"{index}_S*_L*_R1_001.fastq.gz"))
    pairs = []
    for p1 in r1:
        p2 = Path(str(p1).replace("_R1_", "_R2_"))
        if p2.exists():
            pairs.append((p1, p2))
    return pairs

def discover_legacy(key):
    if LEGACY_DIR is None or not LEGACY_DIR.exists():
        return []
    r1 = sorted(LEGACY_DIR.rglob(f"*{key}*R1*.fastq.gz"))
    pairs = []
    for p1 in r1:
        candidates = [
            Path(str(p1).replace("_R1_", "_R2_")),
            Path(str(p1).replace("R1", "R2")),
        ]
        p2 = next((x for x in candidates if x.exists()), None)
        if p2 is not None:
            pairs.append((p1, p2))
    return pairs

def link_pairs(library_id, pairs):
    # Remove only links for this library, never raw files.
    for old in (ROOT/"work"/"inputs").glob(f"{library_id}_L*_R*.fastq.gz"):
        old.unlink()
    for lane, (p1, p2) in enumerate(pairs, start=1):
        for read, src in [(1,p1),(2,p2)]:
            dst = ROOT/"work"/"inputs"/f"{library_id}_L{lane:03d}_R{read}.fastq.gz"
            dst.symlink_to(src.resolve())

resolved = []
missing_rows = []
for _, r in manifest.iterrows():
    pairs = discover_new(r.sequencing_index) if r.batch == "new" else discover_legacy(r.fastq_key)
    if pairs:
        link_pairs(r.library_id, pairs)
    else:
        missing_rows.append({
            "library_id": r.library_id,
            "raw_sample_id": r.raw_sample_id,
            "issue": "missing_fastq",
            "note": "No complete R1/R2 FASTQ pair was found in the configured raw-data directory."
        })
    resolved.append({**r.to_dict(), "fastq_ready": bool(pairs), "n_fastq_pairs": len(pairs)})

resolved = pd.DataFrame(resolved)
resolved.to_csv(ROOT/"results"/"sample_manifest_resolved.csv", index=False)

if missing_rows:
    issues2 = pd.concat([issues, pd.DataFrame(missing_rows)], ignore_index=True).drop_duplicates()
    issues2.to_csv(ROOT/"results"/"metadata_issues.csv", index=False)

print("FASTQ-ready libraries:", int(resolved.fastq_ready.sum()), "/", len(resolved))
print(resolved.groupby(["batch","fastq_ready"]).size())


## MiXCR

The default preset is `milab-human-dna-tcr-multiplex`. Change `MEASLES_MIXCR_PRESET` if the confirmed library protocol uses another preset.

Parallelism is deliberately simple: each MiXCR process receives a fixed number of threads, and only `floor(total CPUs / threads per sample)` libraries are run at the same time.

In [ ]:

if shutil.which("mixcr") is None:
    raise RuntimeError("MiXCR is not available in PATH. Activate the conda environment from environment.yml.")

pilot_ids = ["new_10", "new_11", "new_12", "legacy_UDP0107", "legacy_UDP0108"]
run_table = resolved[resolved.fastq_ready].copy()
if PILOT:
    run_table = run_table[run_table.library_id.isin(pilot_ids)].copy()

max_workers = max(1, CPUS // max(1, THREADS_PER_SAMPLE))
print(f"MiXCR preset: {PRESET}")
print(f"CPUs={CPUS}, threads/sample={THREADS_PER_SAMPLE}, concurrent samples={max_workers}")
print(f"Libraries selected: {len(run_table)}")

def run_mixcr(row):
    lib = row.library_id
    final_tsv = ROOT/"results"/"clonotypes"/f"{lib}.tsv"
    if final_tsv.exists() and final_tsv.stat().st_size > 100:
        return lib, "skipped_existing"

    input_dir = ROOT/"work"/"inputs"
    r1_pattern = str(input_dir / f"{lib}_L{{{{n}}}}_R1.fastq.gz")
    r2_pattern = str(input_dir / f"{lib}_L{{{{n}}}}_R2.fastq.gz")
    prefix = ROOT/"results"/"mixcr"/lib

    cmd = [
        "mixcr", "analyze", PRESET,
        "--threads", str(THREADS_PER_SAMPLE),
        r1_pattern, r2_pattern, str(prefix)
    ]
    log_file = ROOT/"logs"/f"mixcr_{lib}.log"
    with log_file.open("w", encoding="utf-8") as log:
        subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, check=True, cwd=ROOT)

    clns = ROOT/"results"/"mixcr"/f"{lib}.clns"
    if not clns.exists():
        candidates = sorted((ROOT/"results"/"mixcr").glob(f"{lib}*.clns"))
        if not candidates:
            raise RuntimeError(f"MiXCR finished for {lib}, but no .clns file was produced.")
        clns = candidates[0]

    export_cmd = [
        "mixcr", "exportClones",
        "--dont-split-files",
        "--chains", "TRB",
        "--filter-out-of-frames",
        "--filter-stops",
        str(clns), str(final_tsv),
    ]
    with log_file.open("a", encoding="utf-8") as log:
        subprocess.run(export_cmd, stdout=log, stderr=subprocess.STDOUT, check=True, cwd=ROOT)
    if not final_tsv.exists():
        raise RuntimeError(f"MiXCR exportClones did not produce {final_tsv.name}.")
    return lib, "done"

statuses = {}
with ThreadPoolExecutor(max_workers=max_workers) as pool:
    futures = {pool.submit(run_mixcr, row): row.library_id for row in run_table.itertuples()}
    for future in as_completed(futures):
        lib = futures[future]
        try:
            lib, status = future.result()
            statuses[lib] = status
            print(lib, status)
        except Exception as exc:
            statuses[lib] = f"FAILED: {exc}"
            print(lib, statuses[lib])
            raise


## Compact library QC

QC uses the exported TRB clonotype table. It reports productive TRB reads represented by assembled clonotypes, the number of exported TRB clonotypes and the largest clone fraction.

In [ ]:

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

qc = []
for _, r in resolved.iterrows():
    path = ROOT/"results"/"clonotypes"/f"{r.library_id}.tsv"
    if not path.exists():
        continue
    df = pd.read_csv(path, sep="\t")
    count_col = pick_col(df, ["cloneCount","readCount","count"])
    frac_col = pick_col(df, ["cloneFraction","readFraction","fraction"])
    aa_col = pick_col(df, ["aaSeqCDR3","cdr3aa","cdr3_aa"])

    if count_col is None or aa_col is None:
        raise ValueError(f"Cannot identify count/CDR3 columns in {path.name}: {list(df.columns)}")

    counts = pd.to_numeric(df[count_col], errors="coerce").fillna(0)
    total = float(counts.sum())
    if frac_col is not None:
        top_frac = float(pd.to_numeric(df[frac_col], errors="coerce").fillna(0).max())
    else:
        top_frac = float(counts.max() / total) if total > 0 else np.nan

    qc.append({
        "library_id": r.library_id,
        "batch": r.batch,
        "patient_id": r.patient_id,
        "age_group": r.age_group,
        "disease_status": r.disease_status,
        "analysis_timepoint": r.analysis_timepoint,
        "include": r.include,
        "productive_trb_reads": int(total),
        "trb_clonotypes": int(df[aa_col].notna().sum()),
        "top_clone_fraction": top_frac,
    })

qc = pd.DataFrame(qc)
qc.to_csv(ROOT/"results"/"sample_qc.csv", index=False)
qc.head()


In [ ]:

if not qc.empty:
    plt.figure(figsize=(7,5))
    sns.scatterplot(data=qc, x="productive_trb_reads", y="trb_clonotypes", hue="batch", style="disease_status", s=70)
    plt.xlabel("Productive TRB reads in assembled clonotypes")
    plt.ylabel("TRB clonotypes")
    plt.title("Library QC: sequencing depth and recovered repertoire")
    plt.tight_layout()
    for ext in ["png","pdf"]:
        plt.savefig(ROOT/"figures"/f"00_library_qc.{ext}", dpi=200, bbox_inches="tight")
    plt.show()


## UDP0107 versus UDP0108 check

UDP0108 is kept in preprocessing for QC, but remains excluded from biological analyses according to the previous project decision.

In [ ]:

def cdr3_set(lib):
    path = ROOT/"results"/"clonotypes"/f"{lib}.tsv"
    if not path.exists():
        return None, None
    df = pd.read_csv(path, sep="\t")
    aa_col = pick_col(df, ["aaSeqCDR3","cdr3aa","cdr3_aa"])
    count_col = pick_col(df, ["cloneCount","readCount","count"])
    seqs = set(df[aa_col].dropna().astype(str))
    reads = int(pd.to_numeric(df[count_col], errors="coerce").fillna(0).sum())
    return seqs, reads

s7, reads7 = cdr3_set("legacy_UDP0107")
s8, reads8 = cdr3_set("legacy_UDP0108")

if s7 is not None and s8 is not None:
    jaccard = len(s7 & s8) / len(s7 | s8) if (s7 | s8) else np.nan
    pd.DataFrame([{
        "sample_a":"legacy_UDP0107",
        "sample_b":"legacy_UDP0108",
        "productive_reads_a":reads7,
        "productive_reads_b":reads8,
        "cdr3aa_jaccard":jaccard,
        "decision":"UDP0108 excluded from biological analysis"
    }]).to_csv(ROOT/"results"/"udp0107_udp0108_qc.csv", index=False)

    fig, axes = plt.subplots(1,2, figsize=(9,4))
    axes[0].bar(["UDP0107","UDP0108"], [reads7, reads8])
    axes[0].set_ylabel("Productive TRB reads")
    axes[0].set_title("Sequencing depth")
    axes[1].bar(["CDR3aa Jaccard"], [jaccard])
    axes[1].set_ylim(0,1)
    axes[1].set_ylabel("Jaccard overlap")
    axes[1].set_title("Exact CDR3aa overlap")
    fig.suptitle("Legacy control QC: UDP0107 vs UDP0108")
    fig.tight_layout()
    for ext in ["png","pdf"]:
        fig.savefig(ROOT/"figures"/f"00_udp0107_udp0108_qc.{ext}", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("UDP0107/UDP0108 QC will be produced when both libraries have MiXCR exports.")
